<!-- DRAFT: placeholder prose; the author will rewrite this. -->
# 05 · Evaluation
Retrieval scores on the frozen 30 questions (`make eval`), then answer scores: Claude Opus 5.5 answers from each method's top 5, Claude Fable 5.2 judges each required fact by meaning, three runs (`make answers-run`, `make answers-report`). Tables are read from `data/eval/scores/`.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '..'); sys.path.insert(0, '../scripts')
from IPython.display import Markdown
from pathlib import Path
Markdown(Path('../data/eval/scores/latest.md').read_text())

# Retrieval scores

Run 2026-11-18T17:17:59 · commit 284abe1 · gold sha256 5935c443e31e · k=10, entries=5, decay=0.8

### Headline (higher is better, except stale and trap)

| method | recall@5 · all (n=27) | recall@10 · all (n=27) | recall@5 · multi_hop + checklist (n=12) | all_gold@5 · multi_hop + checklist (n=12) | mrr@10 · all (n=27) | ndcg@5 · all (n=27) | stale@5 · all (n=27) | trap@5 · all (n=27) |
|---|---|---|---|---|---|---|---|---|
| M1 BM25 | 0.63 | 0.75 | 0.50 | 0.25 | 0.69 | 0.58 | 0.78 | 0.26 |
| M2 Vector | 0.77 | 0.84 | 0.66 | 0.42 | 0.73 | 0.68 | 0.78 | 0.41 |
| M3 Vector + filter | 0.80 | 0.88 | 0.71 | 0.50 | 0.79 | 0.74 | 0.00 | 0.37 |
| M4 Graph: structural | 0.80 | 0.88 | 0.72 | 0.50 | 0.79 | 0.74 | 0.00 | 0.37 |
| M5 Graph: semantic | 0.80 | 0.88 | 0.71 | 0.50 | 0.79 | 0.74 | 0.00 | 0.33 |
| M6 Graph: both layers | 0.80 | 0.88 | 0.72 | 0.50 | 0.79 | 0.74 | 0.00 | 0.37 |

### Recall@5 by question type

| type | n | M1 | M2 | M3 | M4 | M5 | M6 |
|---|---|---|---|---|---|---|---|
| exact | 3 | 1.00 | 0.67 | 0.67 | 0.67 | 0.67 | 0.67 |
| semantic | 3 | 0.33 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| multi_hop | 7 | 0.60 | 0.79 | 0.83 | 0.90 | 0.83 | 0.90 |
| temporal | 5 | 0.80 | 0.80 | 0.80 | 0.80 | 0.80 | 0.80 |
| conditional | 4 | 0.75 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| checklist | 5 | 0.37 | 0.47 | 0.54 | 0.47 | 0.54 | 0.47 |

### Per question: where each step changed Recall@5

**M1 → M2:** 8 better, 2 worse, 17 unchanged

- q10 (semantic) +1.00: gained inherited_car__2026
- q16 (semantic) +1.00: gained toll_tags_personal_data__2022
- q19 (multi_hop) +1.00: gained epayoff_network__2026, payoff_letter__2026
- q21 (multi_hop) +0.33: gained epayoff_network__2026
- q23 (multi_hop) +0.33: gained selling_with_loan__2022
- q36 (conditional) +1.00: gained home_pickup__2026
- q41 (checklist) +0.20: gained inherited_car__2026
- q42 (checklist) +0.33: gained leased_car__2022
- q04 (exact) -1.00: lost lien_still_on_title__2022
- q24 (multi_hop) -0.33: gained inherited_car__2026; lost epayoff_network__2026, selling_with_loan__2022

**M2 → M3:** 2 better, 0 worse, 25 unchanged

- q24 (multi_hop) +0.33: gained selling_with_loan__2022
- q42 (checklist) +0.33: gained appointment_checklist__2026

**M3 → M4:** 1 better, 1 worse, 25 unchanged

- q22 (multi_hop) +0.50: gained offer_validity__2026; path: offer_validity__2026 via duplicate_title_north_carolina__2022 > offer_validity__2026
- q42 (checklist) -0.33: lost leased_car__2022

**M3 → M5:** 0 better, 0 worse, 27 unchanged


**M3 → M6:** 1 better, 1 worse, 25 unchanged

- q22 (multi_hop) +0.50: gained offer_validity__2026; path: offer_validity__2026 via duplicate_title_north_carolina__2022 > offer_validity__2026
- q42 (checklist) -0.33: lost leased_car__2022



In [2]:
Markdown(Path('../data/eval/scores/answers.md').read_text())

# Answer eval

Mean over the runs, with the lowest and highest run in brackets. facts and all facts: the 27 answerable questions. cites stale / cites trap: share of answers citing an outdated version or a trap article (lower is better). says not covered: answerable questions where the answer said the help center doesn't cover (part of) it, usually because retrieval missed an article (lower is better). declines: the 3 questions the help center doesn't fully cover.

| method | runs | facts found | all facts | says not covered | cites stale | cites trap | declines when not covered |
|---|---|---|---|---|---|---|---|
| M1 BM25 | 3 | 0.65 [0.64–0.66] | 0.53 [0.52–0.56] | 0.48 [0.48–0.48] | 0.16 [0.13–0.17] | 0.13 [0.13–0.13] | 1.00 [1.00–1.00] |
| M2 Vector | 3 | 0.82 [0.80–0.84] | 0.73 [0.70–0.78] | 0.35 [0.30–0.37] | 0.04 [0.03–0.07] | 0.19 [0.17–0.20] | 1.00 [1.00–1.00] |
| M3 Vector + filter | 3 | 0.84 [0.82–0.86] | 0.78 [0.74–0.81] | 0.31 [0.26–0.33] | 0.00 [0.00–0.00] | 0.19 [0.17–0.20] | 1.00 [1.00–1.00] |
| M4 Graph: structural | 3 | 0.85 [0.82–0.88] | 0.80 [0.74–0.85] | 0.30 [0.26–0.33] | 0.00 [0.00–0.00] | 0.19 [0.17–0.20] | 1.00 [1.00–1.00] |
| M5 Graph: semantic | 3 | 0.83 [0.80–0.84] | 0.75 [0.70–0.78] | 0.31 [0.26–0.33] | 0.00 [0.00–0.00] | 0.16 [0.13–0.17] | 1.00 [1.00–1.00] |
| M6 Graph: both layers | 3 | 0.85 [0.82–0.88] | 0.80 [0.74–0.85] | 0.30 [0.26–0.33] | 0.00 [0.00–0.00] | 0.19 [0.17–0.20] | 1.00 [1.00–1.00] |

Answers: claude-opus-5-5 (effort medium), top 5 articles, text only. Judge: claude-fable-5-2 (effort low).

Spend (ledger, from 2026-11-18):
```
  answers                claude-opus-5-5     327 calls  $  3.51
  extract_semantic       claude-opus-5-5      68 calls  $  0.85
  judge                  claude-fable-5-2    327 calls  $  2.65
  total $7.01 of $50.00
```
